# Module 09 - 실습 2: TokenBucket Rate Limiter

## 학습 목표
- 토큰 버킷 알고리즘의 개념을 이해한다
- `TokenBucket` 클래스를 구현할 수 있다
- Rate Limiter로 API 호출 속도를 제어할 수 있다

## 참조 자료
- 학습 문서: `docs/part4-production/09-external-system-integration.md` (섹션 2.3)

## 개념 요약: 토큰 버킷 알고리즘

```
토큰 버킷 비유:
- 양동이(버킷)에 토큰이 있음
- API 호출 시 토큰 1개를 소비
- 일정 속도(rate)로 토큰이 보충됨
- 토큰이 없으면 대기

┌─────────┐
│ ● ● ● ● │ ← 버킷 (최대 capacity개)
│ ● ● ● ● │
│ ● ●     │ ← 현재 6개
└─────────┘
     │
     │ rate개/초씩 보충
     ▼
  API 호출 → 토큰 1개 소비
```

핵심 메서드:
- `__init__(rate, capacity)`: 초당 보충 속도, 최대 용량 설정
- `acquire()`: 토큰 1개 획득 (없으면 대기)
- `_refill()`: 경과 시간에 따라 토큰 보충

In [ ]:
import sys
sys.path.insert(0, '..')

import asyncio
import time

print("Rate Limiter 실습 준비 완료")

## 실습 1: TokenBucket 클래스 구현

`TokenBucket` 클래스를 구현하세요. 다음 속성과 메서드가 필요합니다:
- `__init__(self, rate: float, capacity: int)`: rate=초당 토큰 보충 속도, capacity=최대 용량
- `acquire(self) -> None`: 토큰 1개 획득 (없으면 대기)
- `_refill(self) -> None`: 경과 시간에 따라 토큰 보충

In [ ]:
# TODO: 여기에 코드를 작성하세요
# 힌트 1: __init__에서 self.tokens = capacity (처음에는 가득 참)
# 힌트 2: self.last_refill = time.monotonic() 으로 시간 추적
# 힌트 3: _refill에서 elapsed = now - self.last_refill, tokens += elapsed * self.rate
# 힌트 4: acquire에서 토큰이 없으면 wait_time = (1 - self.tokens) / self.rate 만큼 대기

class TokenBucket:
    """토큰 버킷 기반 Rate Limiter.
    
    초당 허용 요청 수를 제한합니다.
    
    Args:
        rate: 초당 토큰 보충 속도 (예: 5 = 초당 5개)
        capacity: 버킷 최대 용량
    """
    
    def __init__(self, rate: float, capacity: int):
        # TODO: 초기화 구현
        pass
    
    async def acquire(self) -> None:
        """토큰을 1개 획득합니다. 없으면 대기합니다."""
        # TODO: acquire 구현
        # asyncio.Lock()을 사용하세요
        pass
    
    async def _refill(self) -> None:
        """경과 시간에 따라 토큰을 보충합니다."""
        # TODO: _refill 구현
        # self.tokens = min(self.capacity, self.tokens + elapsed * self.rate)
        pass

In [ ]:
# 검증 셀 - TokenBucket 기본 동작 테스트
bucket = TokenBucket(rate=10, capacity=5)
assert hasattr(bucket, 'rate'), "rate 속성이 없습니다"
assert hasattr(bucket, 'capacity'), "capacity 속성이 없습니다"
assert hasattr(bucket, 'tokens'), "tokens 속성이 없습니다"
assert bucket.tokens == 5, f"초기 토큰이 capacity(5)여야 합니다. 현재: {bucket.tokens}"
print("✅ 실습 1-1 완료! TokenBucket 초기화 성공.")

## 실습 2: acquire() 테스트

TokenBucket을 사용하여 rate=5(초당 5개)로 10개의 API 호출을 실행하세요.
처음 5개는 즉시 실행되고, 이후 5개는 토큰이 보충될 때까지 대기해야 합니다.

In [ ]:
# TODO: 여기에 코드를 작성하세요
# 힌트 1: TokenBucket(rate=5, capacity=5)로 생성
# 힌트 2: for 루프로 10번 acquire() 호출
# 힌트 3: time.time()으로 각 호출 시간 기록

limiter = TokenBucket(rate=5, capacity=5)
call_times = []

start = time.time()
for i in range(10):
    # TODO: acquire() 호출 후 현재 시간 기록
    pass

total_time = time.time() - start
print(f"10개 API 호출 완료: {total_time:.2f}초")
print(f"호출 시간: {[f'{t:.2f}s' for t in call_times]}")

In [ ]:
# 검증 셀
assert len(call_times) == 10, f"10번 호출해야 합니다. 현재: {len(call_times)}"
# rate=5이면 5개는 즉시, 나머지 5개는 약 1초 후에 실행
assert total_time >= 0.8, f"rate limiter가 동작하지 않습니다. 소요 시간: {total_time:.2f}초"
print(f"✅ 실습 2 완료! Rate Limiter가 올바르게 동작합니다. (총 {total_time:.2f}초)")

## 실습 3: safe_api_call 함수 작성

Rate Limiter를 적용한 안전한 API 호출 함수를 작성하세요.

In [ ]:
# TODO: Rate Limiter를 사용한 안전한 API 호출 함수
# 힌트 1: 함수 시작 시 rate_limiter.acquire()를 먼저 호출
# 힌트 2: 그 다음 실제 API 호출 (여기서는 asyncio.sleep으로 시뮬레이션)

api_rate_limiter = TokenBucket(rate=3, capacity=3)  # 초당 3개 허용

async def safe_api_call(url: str) -> str:
    """Rate Limit을 지키며 API를 호출합니다."""
    # TODO: rate limiter 적용 후 API 호출
    pass

# 5개 URL에 동시 요청 (rate limit이 적용되어야 함)
urls = [f"https://api.example.com/data/{i}" for i in range(5)]
start = time.time()
responses = await asyncio.gather(*[safe_api_call(url) for url in urls])
elapsed = time.time() - start

print(f"5개 API 호출 완료: {elapsed:.2f}초")
print(f"응답: {responses}")

In [ ]:
# 검증 셀
assert len(responses) == 5, f"응답이 5개여야 합니다. 현재: {len(responses)}"
assert all(r is not None for r in responses), "모든 응답이 None이 아니어야 합니다"
# rate=3이면 3개는 즉시, 나머지 2개는 잠시 대기 후 처리
assert elapsed >= 0.5, f"Rate limiter가 동작하지 않습니다. 소요 시간: {elapsed:.2f}초"
print(f"✅ 실습 3 완료! safe_api_call이 Rate Limit을 지킵니다. (총 {elapsed:.2f}초)")

## 정리

이번 실습에서 배운 내용:
1. 토큰 버킷 알고리즘으로 API 호출 속도를 사전에 제한할 수 있다
2. `rate`(초당 보충)와 `capacity`(최대 용량)로 Rate Limit을 조절한다
3. `asyncio.Lock()`으로 동시성 문제를 방지한다

## 다음 모듈
- **실습 3**: `03_health_check_agent.ipynb` - 시스템 헬스체크 에이전트